In [ ]:
# User Query
#    │
#    ▼
# Input Guard
#    │
#    ▼
# PII Detection & Masking (Presidio)
#    │
#    ▼
# Toxicity Detection (Detoxify)
#    │
#    ▼
# Prompt Injection Detection (Rebuff)
#    │
#    ▼
# Rate Limit Guard
#    │
#    ▼
# Intent Router
#    │
#    ├── Billing Path
#    │       │
#    │       ▼
#    │   Account Lookup
#    │       │
#    │       ▼
#    │   RAG Retriever
#    │       │
#    │       ▼
#    │   Billing LLM
#    │
#    └── Escalate
#            │
#            ▼
#           END

In [ ]:
import os
from typing import TypedDict, Optional,  List, Any
from langgraph.graph import StateGraph, END
# from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

from detoxify import Detoxify
# from rebuff import Rebuff

from langsmith import Client
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults

# from nemoguardrails import LLMRails, RailsConfig

from transformers import pipeline


/home/ggupte/practice/ds/deep_learning/nlp/hug_face/venv_hf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-07 11:31:32.196230: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 11:31:32.270713: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 11:31:33.640167: I tensorflow/core/util/port.cc:153] oneDNN custom op

In [ ]:
# ============================================================
# Environment Configuration
# ============================================================
# Tavily key must be set externally
#assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not set"



In [ ]:
# ============================================================
# vLLM Connection (OpenAI-Compatible)
# ============================================================

# vLLM must be running separately:
# python -m vllm.entrypoints.openai.api_server \
#     --model mistralai/Mistral-7B-Instruct-v0.2
      --port 8002
# llm = ChatOpenAI(
#     base_url="http://localhost:8002/v1",
#     api_key="EMPTY",
#     #model="microsoft/Phi-3-mini-4k-instruct",
#     model="mistralai/Mistral-7B-Instruct-v0.2",
#     temperature=0
#     #, max_tokens=256,
# )

# hf_llm = HuggingFaceEndpoint(
#     #repo_id="mistralai/Mistral-7B-Instruct-v0.2",
#     #task="text-generation",  # this did not work and hence changed to coversation as suggested in the error
#     #task="conversational",
#     repo_id="google/flan-t5-large",
#     task="text-generation",
#     temperature=0.2,
#     max_new_tokens=256,
#     do_sample=False
# )

# # parser = StrOutputParser()

# # llm = hf_llm | parser

# class HFChatWrapper(ChatModel):
#     def __init__(self):
#         self.llm = HuggingFaceEndpoint(
#             repo_id="google/flan-t5-large",
#             task="text-generation",
#             temperature=0.2,
#             max_new_tokens=256,
#         )

#     def _generate(self, messages, stop=None):
#         # Convert chat messages to plain text prompt
#         prompt = ""
#         for m in messages:
#             prompt += f"{m.type.upper()}: {m.content}\n"

#         response = self.llm.invoke(prompt)
#         return AIMessage(content=response)

# llm = HFChatWrapper()

# from langchain_huggingface import ChatHuggingFace

# hf_endpoint = HuggingFaceEndpoint(
#     #repo_id="google/flan-t5-large",
#     #task="text-generation",
#     repo_id="HuggingFaceH4/zephyr-7b-beta",
#     task="conversational",
#     temperature=0.2,
#     max_new_tokens=256,
# )

# llm = ChatHuggingFace(llm=hf_endpoint)


# from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    #api_key="",
    model="openai/gpt-oss-safeguard-20b",
    temperature=0.2
)

In [ ]:
# --------------------------------------------------
# MOCK ACCOUNT DATABASE
# --------------------------------------------------

ACCOUNTS = {
    "user_123": {
        "name": "Alice Johnson",
        "email": "alice@example.com",
        "plan": "Pro",
        "balance": 150.0,
        "billing_date": "2025-01-15"
    }
}

In [ ]:
# --------------------------------------------------
# VECTOR DATABASE
# --------------------------------------------------

docs = [
    "Duplicate charges may happen because payment retries occur.",
    "You may cancel your subscription in the billing settings.",
    "Refunds typically take five to ten business days."
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.create_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(documents, embeddings)

retriever = vector_db.as_retriever(search_kwargs={"k": 2})

/tmp/ipykernel_3301/3759971673.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [ ]:
# --------------------------------------------------
# PII DETECTION
# --------------------------------------------------

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

In [ ]:
# --------------------------------------------------
# TOXICITY MODEL
# --------------------------------------------------

toxicity_model = Detoxify('original')

In [ ]:
# --------------------------------------------------
# GRAPH STATE
# --------------------------------------------------

class SupportState(TypedDict):

    user_id: str
    question: str
    model_calls: int
    intent: Optional[str]
    account_info: Optional[str]
    rag_context: Optional[str]
    response: Optional[str]
    escalate: bool


In [ ]:
# # --------------------------------------------------
# # PROMPT INJECTION DETECTOR
# # --------------------------------------------------

# # rebuff = Rebuff()



# guardrails_config = RailsConfig.from_content(
#     """
#     define user intent injection_attempt
#         "ignore previous instructions"
#         "reveal system prompt"
#         "bypass security"
#         "act as system"

#     define flow detect_injection
#         user intent injection_attempt
#         bot say "Potential prompt injection detected."
#     """)

# guardrails = LLMRails(guardrails_config)


# # --------------------------------------------------
# # PROMPT INJECTION DETECTOR (NeMo Guardrails)
# # --------------------------------------------------

# def injection_guard(state: SupportState):

#     prompt = state["question"]

#     response = guardrails.generate(prompt)

#     if "Potential prompt injection detected" in response:
#         return {
#             "response": "Prompt injection attempt detected.",
#             "escalate": True
#         }

#     return {}

injection_detector = pipeline(
    "text-classification",
    model="protectai/deberta-v3-base-prompt-injection"
)



Device set to use cuda:0


In [ ]:
# --------------------------------------------------
# 1 INPUT GUARD
# --------------------------------------------------

def input_guard(state: SupportState):

    if len(state["question"]) > 500:
        raise ValueError("Query too long")

    return {}

In [ ]:
# --------------------------------------------------
# 2 PII MASKING (Presidio)
# --------------------------------------------------

def pii_mask(state: SupportState):

    text = state["question"]

    results = analyzer.analyze(
        text=text,
        language='en'
    )

    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=results
    )

    return {"question": anonymized.text}

In [ ]:
# --------------------------------------------------
# 3 TOXICITY FILTER (Detoxify)
# --------------------------------------------------

def toxicity_filter(state: SupportState):

    result = toxicity_model.predict(state["question"])

    if result["toxicity"] > 0.6:
        return {
            "response": "Please avoid abusive language.",
            "escalate": True
        }

    return {}

In [ ]:
# # --------------------------------------------------
# # 4 PROMPT INJECTION DETECTOR (Rebuff)
# # --------------------------------------------------

# def injection_guard(state: SupportState):

#     detection = injection_guard(state["question"])

#     if detection.is_injection:
#         return {
#             "response": "Prompt injection attempt detected.",
#             "escalate": True
#         }

#     return {}

def injection_guard(state: SupportState):

    result = injection_detector(state["question"])[0]

    label = result["label"]
    score = result["score"]

    if label == "INJECTION" and score > 0.7:
        return {
            "response": "Prompt injection attempt detected.",
            "escalate": True
        }

    return {}

In [ ]:
# --------------------------------------------------
# 5 RATE LIMIT
# --------------------------------------------------

def rate_limit_guard(state: SupportState):

    if state["model_calls"] >= 3:
        return {"escalate": True}

    return {"model_calls": state["model_calls"] + 1}

In [ ]:
# --------------------------------------------------
# 6 INTENT ROUTER
# --------------------------------------------------

def detect_intent(state: SupportState):

    prompt = f"""
Classify the query into one word.

billing
other

Query: {state['question']}
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    intent = result.content.strip().lower()

    return {"intent": intent}

In [ ]:
# --------------------------------------------------
# 7 ACCOUNT LOOKUP
# --------------------------------------------------

def account_lookup(state: SupportState):

    user = ACCOUNTS.get(state["user_id"])

    if not user:
        return {"account_info": "Account not found"}

    info = f"""
Name: {user['name']}
Email: {user['email']}
Plan: {user['plan']}
Balance: {user['balance']}
Billing Date: {user['billing_date']}
"""

    return {"account_info": info}

In [ ]:
# --------------------------------------------------
# 8 RAG RETRIEVAL
# --------------------------------------------------

def rag_retrieval(state: SupportState):

    docs = retriever.invoke(state["question"])
    context = "\n".join([d.page_content for d in docs])
    return {"rag_context": context}

In [ ]:
# --------------------------------------------------
# 9 BILLING AGENT
# --------------------------------------------------

def billing_agent(state: SupportState):

    prompt = f"""
You are a billing support agent.

User Question:
{state['question']}

Account Info:
{state['account_info']}

Knowledge Base:
{state['rag_context']}

Answer clearly.
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {"response": result.content}


In [ ]:
# --------------------------------------------------
# ESCALATE
# --------------------------------------------------

def escalate(state: SupportState):

    print("🚨 Escalated to human support")

    return {
        "response": "Your issue has been escalated to a human support agent."
    }

In [ ]:
# --------------------------------------------------
# ROUTERS
# --------------------------------------------------

def route_escalation(state: SupportState):

    if state.get("escalate"):
        return "escalate"

    return "intent_router"


def route_intent(state: SupportState):

    if state["intent"] == "billing":
        return "account_lookup"

    return "escalate"


In [ ]:
# --------------------------------------------------
# BUILD GRAPH
# --------------------------------------------------

builder = StateGraph(SupportState)

builder.add_node("input_guard", input_guard)
builder.add_node("pii_mask", pii_mask)
builder.add_node("toxicity_filter", toxicity_filter)
builder.add_node("injection_guard", injection_guard)
builder.add_node("rate_limit", rate_limit_guard)
builder.add_node("intent_router", detect_intent)
builder.add_node("account_lookup", account_lookup)
builder.add_node("rag_retrieval", rag_retrieval)
builder.add_node("billing_agent", billing_agent)
builder.add_node("escalate", escalate)


builder.set_entry_point("input_guard")

builder.add_edge("input_guard", "pii_mask")
builder.add_edge("pii_mask", "toxicity_filter")
builder.add_edge("toxicity_filter", "injection_guard")
builder.add_edge("injection_guard", "rate_limit")


builder.add_conditional_edges(
    "rate_limit",
    route_escalation,
    {
        "intent_router": "intent_router",
        "escalate": "escalate"
    }
)

builder.add_conditional_edges(
    "intent_router",
    route_intent,
    {
        "account_lookup": "account_lookup",
        "escalate": "escalate"
    }
)

builder.add_edge("account_lookup", "rag_retrieval")
builder.add_edge("rag_retrieval", "billing_agent")

builder.add_edge("billing_agent", END)
builder.add_edge("escalate", END)


graph = builder.compile()

In [ ]:

# --------------------------------------------------
# RUN EXAMPLE
# --------------------------------------------------

result = graph.invoke(
    {
        "user_id": "user_123",
        "question": "Why was I charged twice this month?",
        "model_calls": 0,
        "intent": None,
        "account_info": None,
        "rag_context": None,
        "response": None,
        "escalate": False
    }
)

print("\nFINAL RESPONSE\n")
print(result["response"])


FINAL RESPONSE

Hi Alice,

I’m sorry you’re seeing a duplicate charge.  
In our system a second charge can appear if a payment retry is triggered—usually when the first attempt fails (e.g., a temporary network issue or a declined card). Once the retry succeeds, the original pending transaction is settled, so you’ll see two entries in your statement.

**What to do next**

1. **Check your bank statement** – the second charge should be a refund of the first if the retry was successful.  
2. **Verify your card details** – make sure the card on file is up‑to‑date.  
3. **Cancel or modify your subscription** – if you no longer want the Pro plan, you can do so in the billing settings of your account.

If the duplicate charge remains after a few days or you’re still unsure, let me know and I’ll investigate further.

Thank you for your patience!
